<img src="https://drive.usercontent.google.com/download?id=19NlKhM3FwlX-_9yCEnhZyLDPr4wicWmC&export=view" width="120" align="right"/>

# เอกสารประกอบการเรียนรู้
## หลักสูตร AI & Data Analytics (Machine Learning)
### บทที่ 2 — Classification: จำแนกประเภทด้วย ML

---

**โครงการ Coding Thailand 2026**
ระดับ: ผู้เริ่มต้น | ระยะเวลา: 30 นาที | รูปแบบ: Interactive Notebook

---

**ผู้สอน**

**ผศ.ดร. สัญชัย เอียดปราบ**

อาจารย์ประจำหลักสูตรวิศวกรรมระบบสมองกลฝังตัวและอิเล็กทรอนิกส์สื่อสาร
คณะวิศวกรรมศาสตร์ มหาวิทยาลัยบูรพา

---

**วัตถุประสงค์การเรียนรู้ (Learning Objectives)**

เมื่อศึกษาบทนี้จบแล้ว ผู้เรียนสามารถ:
1. เตรียมข้อมูลสำหรับ ML ได้ (Features / Label / Train-Test Split)
2. สร้าง Train และทดสอบ Classification Model ได้
3. ประเมินผลโมเดลด้วย Accuracy และ Confusion Matrix ได้
4. เปรียบเทียบ Model หลายตัวเพื่อเลือกตัวที่เหมาะสมได้

**โจทย์ประจำบท:** สร้าง AI จำแนกสายพันธุ์เพนกวิน 🐧 จากขนาดตัว

---
## หัวข้อที่ 1 — โหลดและสำรวจข้อมูล
**เวลาโดยประมาณ: 8 นาที**

ขั้นตอนแรกของ ML Pipeline คือการทำความเข้าใจข้อมูล (Exploratory Data Analysis — EDA) เพื่อดูว่าข้อมูลมีอะไรบ้าง มีค่าหายไปหรือไม่ และข้อมูลแยกกลุ่มได้ชัดเจนเพียงใด

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# โหลด Penguins Dataset (มีใน seaborn พร้อมใช้เลย!)
df = sns.load_dataset('penguins')
print(f"ข้อมูล: {df.shape[0]} ตัว, {df.shape[1]} คอลัมน์")
df.head()

In [ ]:
# ดูข้อมูลเบื้องต้น
print("สายพันธุ์:")
print(df["species"].value_counts())
print(f"\nข้อมูลหายไป:")
print(df.isna().sum())

In [ ]:
# ลบข้อมูลที่หายไป (วิธีง่ายสำหรับเริ่มต้น)
df_clean = df.dropna()
print(f"ข้อมูลหลัง clean: {len(df_clean)} ตัว")

In [ ]:
# Scatter Plot: ดูว่าข้อมูลแยกกลุ่มได้ไหม
plt.figure(figsize=(10, 6))
species_colors = {"Adelie": "#e74c3c", "Chinstrap": "#3498db", "Gentoo": "#2ecc71"}

for species in df_clean["species"].unique():
    data = df_clean[df_clean["species"] == species]
    plt.scatter(data["bill_length_mm"], data["body_mass_g"],
                label=species, color=species_colors[species], alpha=0.7, s=50)

plt.xlabel("Bill Length (mm)")
plt.ylabel("Body Mass (g)")
plt.title("Penguin Species: Bill Length vs Body Mass")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## หัวข้อที่ 2 — เตรียมข้อมูลสำหรับ ML
**เวลาโดยประมาณ: 7 นาที**

ก่อน Train โมเดล ต้องแบ่งข้อมูลออกเป็น 2 ส่วน เพื่อให้สามารถวัดผลได้อย่างยุติธรรม

**คำศัพท์สำคัญ:**

| คำศัพท์ | ความหมาย |
|---------|----------|
| **Features (X)** | ข้อมูลที่ AI ใช้ตัดสินใจ (ความยาวปาก, น้ำหนัก ฯลฯ) |
| **Label (y)** | คำตอบที่เราอยากให้ AI ทำนาย (สายพันธุ์) |
| **Train Set** | ข้อมูลสำหรับสอน AI (80%) |
| **Test Set** | ข้อมูลสำหรับทดสอบ AI (20%) — AI ไม่เคยเห็นมาก่อน! |

In [ ]:
from sklearn.model_selection import train_test_split

# เลือก Features (X) และ Label (y)
X = df_clean[["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]]
y = df_clean["species"]

print(f"Features (X): {X.shape} → {list(X.columns)}")
print(f"Label (y): {y.shape} → {y.unique()}")

In [ ]:
# แบ่ง Train / Test (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {len(X_train)} ตัว (สอน AI)")
print(f"Test:  {len(X_test)} ตัว (ทดสอบ AI)")
print(f"\n💡 AI จะเรียนรู้จาก {len(X_train)} ตัว แล้วเราจะทดสอบว่ามันทายถูกกี่ตัวจาก {len(X_test)} ตัว")

---
## หัวข้อที่ 3 — สร้าง Train และทดสอบ Model
**เวลาโดยประมาณ: 10 นาที**

ในหัวข้อนี้จะใช้ **Decision Tree** ซึ่งเป็นอัลกอริทึมพื้นฐานที่เข้าใจง่าย — ทำงานโดยการตั้งคำถาม Yes/No แบ่งข้อมูลเป็นกลุ่มเล็กลงไปเรื่อย ๆ จนได้คำตอบ

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# สร้าง Model: Decision Tree
model = DecisionTreeClassifier(random_state=42)

# Train: สอน AI ด้วยข้อมูล Train
model.fit(X_train, y_train)

print("✅ Train เสร็จแล้ว!")

In [ ]:
# ทดสอบ: ให้ AI ทำนายข้อมูล Test
y_pred = model.predict(X_test)

# วัดผล
accuracy = accuracy_score(y_test, y_pred)
print(f"🎯 Accuracy: {accuracy:.1%}")
print(f"   (ทายถูก {int(accuracy * len(y_test))}/{len(y_test)} ตัว)")

In [ ]:
# Confusion Matrix: ดูว่าผิดตรงไหน
from sklearn.metrics import ConfusionMatrixDisplay

plt.figure(figsize=(7, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap='Blues')
plt.title(f'Decision Tree — Accuracy: {accuracy:.1%}')
plt.tight_layout()
plt.show()

print("💡 แนวทแยง = ทายถูก, นอกแนวทแยง = ทายผิด")

In [ ]:
# ลองทำนายเพนกวินตัวใหม่!
print("🐧 ลองทำนายเพนกวินตัวใหม่:")
print("-" * 50)

new_penguins = [
    [39.0, 18.5, 186, 3800],   # ปากสั้น ครีบยาว ตัวกลาง
    [48.0, 15.5, 200, 4800],   # ปากยาว ครีบยาว ตัวใหญ่
    [50.0, 19.0, 195, 3600],   # ปากยาวมาก ตัวเล็ก
]

predictions = model.predict(new_penguins)

for i, (penguin, pred) in enumerate(zip(new_penguins, predictions)):
    print(f"  เพนกวิน {i+1}: ปาก {penguin[0]}mm, ครีบ {penguin[2]}mm, น้ำหนัก {penguin[3]}g")
    print(f"  → AI ทำนาย: {pred}\n")

---
## หัวข้อที่ 4 — เปรียบเทียบ Model
**เวลาโดยประมาณ: 5 นาที**

Decision Tree ไม่ใช่ Model เดียวที่ใช้ได้ ลองเปรียบเทียบกับ **KNN (K-Nearest Neighbors)** ซึ่งทำงานโดยดูว่าข้อมูลใหม่ใกล้เคียงกับข้อมูลตัวอย่างกลุ่มไหนมากที่สุด

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Model 2: KNN (K-Nearest Neighbors)
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)
acc_knn = accuracy_score(y_test, y_pred_knn)

print("📊 เปรียบเทียบ Model:")
print(f"  Decision Tree : {accuracy:.1%}")
print(f"  KNN (k=5)     : {acc_knn:.1%}")
print(f"\n  🏆 ชนะ: {'Decision Tree' if accuracy >= acc_knn else 'KNN'}")

In [ ]:
# Bar Chart เปรียบเทียบ
models = ["Decision Tree", "KNN (k=5)"]
scores = [accuracy, acc_knn]

plt.figure(figsize=(6, 4))
bars = plt.bar(models, scores, color=['#3498db', '#e74c3c'])
for bar, score in zip(bars, scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{score:.1%}', ha='center', fontsize=14, fontweight='bold')
plt.ylim(0, 1.1)
plt.ylabel('Accuracy')
plt.title('Model Comparison: Penguin Classification')
plt.tight_layout()
plt.show()

---
## สรุปสาระสำคัญ (Summary Cheat Sheet)

### ML Pipeline สำหรับ Classification

```python
# 1. เตรียมข้อมูล
X = df[["feature1", "feature2"]]   # Features
y = df["label"]                      # Label
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# 2. สร้าง + Train
model = DecisionTreeClassifier()
model.fit(X_train, y_train)

# 3. ทดสอบ
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))
```

| ขั้นตอน | คำสั่ง |
|---------|--------|
| แบ่งข้อมูล | `train_test_split(X, y, test_size=0.2)` |
| สร้าง Model | `DecisionTreeClassifier()` |
| Train | `model.fit(X_train, y_train)` |
| ทำนาย | `model.predict(X_test)` |
| วัดผล | `accuracy_score(y_test, y_pred)` |

---

**บทถัดไป:** บทที่ 3 — Regression + Clustering: ML ทำได้มากกว่าจำแนก!